# 03 Inference

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"

BEHAVIOR_VAR = "EverCigaretteUse"
BEHAVIOR_P0 = 0.50
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

processed = pd.read_csv(PROCESSED_PATH)
print("Processed data shape:", processed.shape)

Processed data shape: (14041, 3)


## 2.Proportion Inference: `EverCigaretteUse`

In [2]:
behavior_binary = processed["behavior_binary"].dropna().astype(int)
n_b = int(behavior_binary.shape[0])
x_b = int((behavior_binary == 1).sum())
phat = x_b / n_b

ci_b = stats.binomtest(x_b, n_b).proportion_ci(confidence_level=0.95, method="wilson")
z_stat = (phat - BEHAVIOR_P0) / np.sqrt(BEHAVIOR_P0 * (1 - BEHAVIOR_P0) / n_b)
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_stat)))
p_value_display = "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"

prop_results = pd.DataFrame({
    "quantity": [
        "sample_size", "successes", "sample_proportion",
        "benchmark_p0", "z_statistic", "p_value", "ci_low", "ci_high"
    ],
    "value": [
        str(n_b),
        str(x_b),
        f"{phat:.4f}",
        f"{BEHAVIOR_P0:.4f}",
        f"{z_stat:.4f}",
        p_value_display,
        f"{ci_b.low:.4f}",
        f"{ci_b.high:.4f}"
    ]
})

decision_b = "reject H0" if p_value_b < 0.05 else "fail to reject H0"

display(prop_results)

,quantity,value
0,sample_size,13601
1,successes,7164
2,sample_proportion,0.5267
3,benchmark_p0,0.5000
4,z_statistic,6.2337
5,p_value,< 0.0001
6,ci_low,0.5183
7,ci_high,0.5351


### Interpretation (Proportion Analysis)

- The estimated sample proportion is **0.5267**, based on **13,601** valid students.
- The benchmark proportion is **0.5000**, and the **95% confidence interval** is **(0.5183, 0.5351)**.
- Because the **p-value is < 0.0001**, we **reject the null hypothesis** at the 5% significance level.
- In context, this suggests that the population proportion for **EverCigaretteUse** is **above 0.50**.
- This result is consistent with the EDA, where the success category was slightly more common than the failure category.
- It is also broadly consistent with the additional EDA, where the success proportion tended to be higher in the upper weight quartiles.
- A cautious point is that the result is statistically significant, but the difference from **0.50** is still fairly small in absolute terms.

## 3.Mean Inference: `HowMuchDoYouWeighWithoutShoesInKG`

In [3]:
w = processed[CONT_VAR].dropna()
n_w = int(w.shape[0])
mean_w = float(w.mean())
sd_w = float(w.std(ddof=1))
se_w = sd_w / np.sqrt(n_w)

t_crit = stats.t.ppf(0.975, df=n_w - 1)
ci_low_w = mean_w - t_crit * se_w
ci_high_w = mean_w + t_crit * se_w

t_stat, p_value_w = stats.ttest_1samp(w, popmean=CONT_MU0)
p_value_display = "< 0.0001" if p_value_w < 0.0001 else f"{p_value_w:.4f}"

mean_results = pd.DataFrame({
    "quantity": [
        "sample_size", "sample_mean", "sample_sd",
        "benchmark_mu0", "t_statistic", "p_value", "ci_low", "ci_high"
    ],
    "value": [
        str(n_w),
        f"{mean_w:.4f}",
        f"{sd_w:.4f}",
        f"{CONT_MU0:.4f}",
        f"{t_stat:.4f}",
        p_value_display,
        f"{ci_low_w:.4f}",
        f"{ci_high_w:.4f}"
    ]
})

decision_w = "reject H0" if p_value_w < 0.05 else "fail to reject H0"

display(mean_results)

,quantity,value
0,sample_size,13062
1,sample_mean,68.5502
2,sample_sd,16.9909
3,benchmark_mu0,68.0000
4,t_statistic,3.7007
5,p_value,0.0002
6,ci_low,68.2588
7,ci_high,68.8416


### Interpretation (Mean Analysis)

- The sample mean weight is **68.5502 kg**, based on **13,062** valid students, with a sample standard deviation of **16.9909 kg**.
- The benchmark mean is **68.0000 kg**, and the **95% confidence interval** is **(68.2588, 68.8416) kg**.
- Because the **p-value is 0.0002**, we **reject the null hypothesis** at the 5% significance level.
- In context, this suggests that the population mean weight is **slightly above 68.0 kg**.
- This result is consistent with the EDA, where the sample mean was above the benchmark and the histogram showed a mild right tail.
- It is also consistent with the additional EDA, where the success group tended to have a somewhat higher weight distribution than the failure group.
- A cautious point is that the result is statistically significant, but the practical difference from **68.0 kg** is not very large.

## 4.Summary Table of Main Inferential Results

In [4]:
inference_summary = pd.DataFrame({
    "analysis": ["Population proportion", "Population mean"],
    "variable": [BEHAVIOR_VAR, CONT_VAR],
    "estimate": [f"{phat:.4f}", f"{mean_w:.4f}"],
    "benchmark": [f"{BEHAVIOR_P0:.4f}", f"{CONT_MU0:.4f}"],
    "test_statistic": [f"{z_stat:.4f}", f"{t_stat:.4f}"],
    "p_value": [
        "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}",
        "< 0.0001" if p_value_w < 0.0001 else f"{p_value_w:.4f}"
    ],
    "ci_low": [f"{ci_b.low:.4f}", f"{ci_low_w:.4f}"],
    "ci_high": [f"{ci_b.high:.4f}", f"{ci_high_w:.4f}"],
    "decision_5pct": [decision_b, decision_w]
})

inference_summary.to_csv(TAB_DIR / "03_inference_summary_table.csv", index=False)
display(inference_summary)

,analysis,variable,estimate,benchmark,test_statistic,p_value,ci_low,ci_high,decision_5pct
0,Population proportion,EverCigaretteUse,0.5267,0.5000,6.2337,< 0.0001,0.5183,0.5351,reject H0
1,Population mean,HowMuchDoYouWeighWithoutShoesInKG,68.5502,68.0000,3.7007,0.0002,68.2588,68.8416,reject H0


## 5.Final Synthesis

This project asked two question-driven one-sample inference questions using the cleaned YRBS dataset.

For the **behavior variable**, the EDA showed that the success category for `EverCigaretteUse` appeared slightly more common than the failure category. The additional EDA also suggested that the success proportion tended to be higher in the upper weight quartiles. The formal proportion analysis confirmed the overall pattern: the estimated proportion was above the benchmark of **0.50**, the **95% confidence interval** stayed above **0.50**, and the one-sample test rejected the null hypothesis.

For the **continuous variable**, the EDA showed a large sample, moderate spread, and a somewhat right-skewed weight distribution with some possible outliers. The additional EDA also suggested that the success group tended to have a somewhat higher weight distribution than the failure group. Even so, the sample mean was only slightly above **68.0 kg**. The **95% confidence interval** and the one-sample **t-test** both indicated that the mean weight is statistically different from **68.0 kg**, although the practical difference is fairly small.

Overall, the inferential conclusions are consistent with the EDA.

## 6. Output Final Summary & README files

In [5]:
final_summary_text = f'''Project Cycle 2 Final Summary

Selected behavior variable: {BEHAVIOR_VAR}
Benchmark proportion: {BEHAVIOR_P0:.2f}
Estimated proportion: {phat:.6f}
95% CI: ({ci_b.low:.6f}, {ci_b.high:.6f})
p-value: {p_value_b:.6g}
Decision at 5%: {decision_b}

Selected continuous variable: {CONT_VAR}
Benchmark mean: {CONT_MU0:.1f}
Estimated mean: {mean_w:.6f}
95% CI: ({ci_low_w:.6f}, {ci_high_w:.6f})
p-value: {p_value_w:.6g}
Decision at 5%: {decision_w}
'''
(SUM_DIR / "final_summary.txt").write_text(final_summary_text, encoding="utf-8")

readme_text = f'''# Project Cycle 2

## Group Information
- Group number: 09
- Member names: 112370216 蘇榮盛, 109370231 謝宇家, 113370220 游子涵

## Dataset
- `YRBS_2007.csv`, `yrbs_selected_cleaned.csv`

## Selected Variables
- Proportion analysis: `{BEHAVIOR_VAR}` with benchmark `p0 = {BEHAVIOR_P0:.2f}`
- Mean analysis: `{CONT_VAR}` with benchmark `mu0 = {CONT_MU0:.1f}`

## Project Questions
1. Is the proportion of students who have ever used cigarettes different from 0.50?
2. Is the mean body weight of students different from 68.0 kg?

## Short Final Conclusion
Using the provided dataset, the sample proportion for `EverCigaretteUse` is above 0.50, and the sample mean for `HowMuchDoYouWeighWithoutShoesInKG` is slightly above 68.0 kg. Both one-sample tests reject the corresponding null hypothesis at the 5% level.
'''
(ROOT / "README.md").write_text(readme_text, encoding="utf-8")
print("Saved:", SUM_DIR / "final_summary.txt")
print("Saved:", ROOT / "README.md")

Saved: C:\Users\asd45\Downloads\project-cycle-2_v2.1\outputs\summary\final_summary.txt
Saved: C:\Users\asd45\Downloads\project-cycle-2_v2.1\README.md
